# ISOM 835 · Session 3 — Feature Engineering & Leak-Proof Pipelines
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Sep 28 · Prof. Hasan Arslan**

Encoders, scalers, imputers, and the scikit-learn **Pipeline** — the machinery that turns a messy table into a model input without ever peeking at the test set.

> **Frame the prediction (Bank Marketing).** *Unit:* one phone call to a client · *Target:* did the client subscribe to a term deposit? · *Horizon:* before the call · *Decision:* whom the call center dials first.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

In [ ]:
import pandas as pd, numpy as np, sklearn
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
print('scikit-learn', sklearn.__version__)

## 1. The wrong way, live
Scale the whole dataset, *then* split. The score looks fine — and it is subtly a lie, because the scaler learned the test set's mean and variance. For a scaler the leak is small. For an imputer, a target encoder, or a feature selector it can be enormous. **Same habit, same fix: split first, fit everything on the training fold.**

In [ ]:
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
telco = pd.read_csv(URL)
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')
Xn = telco[['tenure', 'MonthlyCharges', 'TotalCharges']].fillna(0)
y = (telco['Churn'] == 'Yes').astype(int)

# ❌ wrong: fit the scaler on ALL rows, then split
Z = StandardScaler().fit_transform(Xn)
Ztr, Zte, ytr, yte = train_test_split(Z, y, test_size=0.2, stratify=y, random_state=835)
wrong = roc_auc_score(yte, KNeighborsClassifier(25).fit(Ztr, ytr).predict_proba(Zte)[:, 1])

# ✅ right: split first; the scaler lives inside a pipeline fit on train only
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.2, stratify=y, random_state=835)
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(25)).fit(Xtr, ytr)
right = roc_auc_score(yte, pipe.predict_proba(Xte)[:, 1])
print(f'AUC wrong-way {wrong:.4f}   right-way {right:.4f}   (tiny gap here — the habit is what matters)')

## 2. Meet the data: Bank Marketing
45,211 phone calls from a Portuguese bank's term-deposit campaign (Moro, Cortez & Rita, 2014). OpenML's copy uses anonymous column names `V1…V16`; the UCI documentation maps them. **`V12` is call duration — known only after the call ends. It is the most famous leak in tabular ML, and we drop it.**

In [ ]:
bank = fetch_openml('bank-marketing', version=1, as_frame=True)
names = ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']
X = bank.data.set_axis(names, axis=1)
y = (bank.target == '2').astype(int)            # '2' = subscribed
print(X.shape, f'subscribe rate {y.mean():.1%}')
X = X.drop(columns=['duration'])                # LEAK: known only after the call
X.dtypes

## 3. Column types → transformers
Numeric columns: impute (median) → scale. Categorical columns: encode. The `ColumnTransformer` routes each column to its branch; the `Pipeline` chains preprocessing to the model. One object, one `fit`, zero leakage.

In [ ]:
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print('numeric:', num_cols); print('categorical:', cat_cols)

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
model = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=2000))])
model     # in Colab this renders as an interactive wiring diagram

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=835)
scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'Logistic regression, one-hot, 5-fold AUC: {scores.mean():.3f} ± {scores.std():.3f}')

Every fold fits the imputer, the scaler, and the encoder on its own training portion — that is what `cross_val_score` on a *Pipeline* guarantees.

## 4. Encoders are choices, not defaults
- **OneHotEncoder** — a 0/1 column per category. Right for a handful of unordered categories (`job`, `contact`).
- **OrdinalEncoder** — integers, *only* when the order is real (`education`: primary < secondary < tertiary). Trees don't mind arbitrary integers; linear models do.
- **TargetEncoder** (scikit-learn ≥ 1.3) — replace the category with the (cross-fitted) mean of the target. Right for many categories. Cross-fitting is what stops it from leaking the label; a hand-rolled `groupby().mean()` on the full data would.
- Always `handle_unknown='ignore'` — production will show you a category you never saw.

In [ ]:
edu_order = [['primary', 'secondary', 'tertiary', 'unknown']]
prep2 = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('edu', OrdinalEncoder(categories=edu_order, handle_unknown='use_encoded_value', unknown_value=-1), ['education']),
    ('tgt', TargetEncoder(random_state=835), ['job', 'month']),          # many categories → target encoding
    ('ohe', OneHotEncoder(handle_unknown='ignore'), [c for c in cat_cols if c not in ('education', 'job', 'month')]),
])
model2 = Pipeline([('prep', prep2), ('clf', LogisticRegression(max_iter=2000))])
scores2 = cross_val_score(model2, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'mixed encoders, 5-fold AUC: {scores2.mean():.3f} ± {scores2.std():.3f}')

## 5. The payoff: swap the model, keep everything else

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
model3 = Pipeline([('prep', prep), ('clf', HistGradientBoostingClassifier(random_state=835))])
scores3 = cross_val_score(model3, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'same pipeline, gradient boosting: {scores3.mean():.3f} ± {scores3.std():.3f}   (Session 8 explains why)')

## 6. When scaling matters — and when it doesn't
Distance- and gradient-based models (kNN, logistic/linear regression, SVM, neural nets) care about scale. Trees and forests split on thresholds and do not. Test it.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
yt = (telco['Churn'] == 'Yes').astype(int)
Xtr, Xte, ytr, yte = train_test_split(telco[['tenure', 'MonthlyCharges', 'TotalCharges']].fillna(0), yt, test_size=0.2, stratify=yt, random_state=835)
for name, clf in [('kNN', KNeighborsClassifier(25)), ('logistic', LogisticRegression(max_iter=2000)), ('tree(depth 5)', DecisionTreeClassifier(max_depth=5, random_state=835))]:
    raw = roc_auc_score(yte, clf.fit(Xtr, ytr).predict_proba(Xte)[:, 1])
    sc = roc_auc_score(yte, make_pipeline(StandardScaler(), clf).fit(Xtr, ytr).predict_proba(Xte)[:, 1])
    print(f'{name:14s} raw {raw:.3f}   scaled {sc:.3f}')

## 7. Feature engineering from domain knowledge (Telco)
A feature is a hypothesis about why customers leave. We add three with `FunctionTransformer` so they live *inside* the pipeline, then measure whether AUC moves.

In [ ]:
from sklearn.preprocessing import FunctionTransformer
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
def engineer(d):
    d = d.copy()
    d['charges_per_month'] = d['TotalCharges'].fillna(0) / d['tenure'].clip(lower=1)      # ratio
    d['new_customer'] = (d['tenure'] <= 6).astype(int)                                       # bucket
    d['n_addons'] = (d[services] == 'Yes').sum(axis=1)                                       # count
    d['m2m_echeck'] = ((d['Contract'] == 'Month-to-month') & (d['PaymentMethod'] == 'Electronic check')).astype(int)  # interaction
    return d
T = telco.drop(columns=['customerID', 'Churn']); yt = (telco['Churn'] == 'Yes').astype(int)
def build(feature_fn):
    Xf = feature_fn(T) if feature_fn else T
    num = Xf.select_dtypes(include='number').columns.tolist(); cat = [c for c in Xf.columns if c not in num]
    prep = ColumnTransformer([('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
    return Xf, Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=3000))])
for label, fn in [('base features', None), ('+ engineered', engineer)]:
    Xf, pipe = build(fn)
    s = cross_val_score(pipe, Xf, yt, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'{label:16s} AUC {s.mean():.4f} ± {s.std():.4f}')

## 8. The 2025 shortcut: skrub's TableVectorizer
`skrub` guesses a sensible encoder for every column (dates, high-cardinality strings, numbers) so a first model is one line. Use it for the *first* model; hand-craft for the one you ship.

In [ ]:
# OPTIONAL — pip install skrub (Colab: !pip install -q skrub)
try:
    from skrub import TableVectorizer
    auto = Pipeline([('tv', TableVectorizer()), ('clf', HistGradientBoostingClassifier(random_state=835))])
    s = cross_val_score(auto, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'TableVectorizer + GBM: {s.mean():.3f} ± {s.std():.3f}')
except ImportError:
    print('skrub not installed — run: pip install skrub')

## 9. Your turn
1. **Add `poutcome`-aware features.** In Bank Marketing, `pdays == -1` means never contacted before. Make an indicator for it and a capped version of `pdays`. Does 5-fold AUC move?
2. **Encoder bake-off.** Encode `job` three ways (one-hot, ordinal, target) in otherwise identical pipelines with logistic regression. Rank them and explain the ranking in one sentence.
3. **Rebuild HW1 as a Pipeline.** Confirm your Telco AUC is unchanged, then add one interaction feature of your own design.

In [ ]:
# Your turn — work here

## What we learned tonight
- **Everything that learns from data is a model** — scaler, imputer, encoder. Fit on the training fold only; let `Pipeline` + `ColumnTransformer` enforce it.
- **Encoders are choices:** one-hot for few categories, ordinal for real order, target encoding (cross-fit) for many. `handle_unknown='ignore'` always.
- **Scale for distance/gradient models, not for trees.**
- **A feature is a hypothesis.** Ratios, buckets, counts, interactions — the ones you invent from domain knowledge usually beat the algorithm choice.

Next week: regression — predicting numbers with the same pipeline, a log target, and ridge/lasso. Read ISLP Chapter 3.